# 3. OVERNIGHT GAP TRADING — ROBUSTNESS BATTERY & VALIDATION GATE
## SYSTEMATIC TRADING APPROACH FOLLOWING ALPHACA FRAMEWORK

Phases:
1. OOS validation (100% Capital + 1% ATR sizing)
2. Walk-forward (3y IS / 1y OOS, fixed rules)
3. Block Monte Carlo (10,000 × block=5)
4. Monkey test (2,000 synthetic portfolios, same duration + eligibility)
5. Stress: costs ×2, exclude top-2 years, crisis regimes
6. Edge decay (rolling 40-trade PF, recent vs historical)
7. Full validation gate (inline)

In [ ]:
# ===== 03 — Robustness battery & validation gate =====
import os, sys, importlib, shutil
from pathlib import Path
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import Markdown, display

def find_upwards(name, start=None, must_be_dir=True):
    here = (start or Path.cwd()).resolve()
    for base in (here, *here.parents):
        target = base / name
        if target.is_dir() if must_be_dir else target.exists():
            return base
    raise FileNotFoundError(f"Could not find '{name}' walking up from {here}")

EDGE_DIR    = find_upwards('variants')
utils_path  = str(find_upwards('utils') / 'utils')
engine_path = str(EDGE_DIR)
engine_dir  = str(EDGE_DIR / 'overnight_gap')
for p in (utils_path, engine_path, engine_dir):
    if p not in sys.path:
        sys.path.insert(0, p)

import overnight_gap_engine as eng
import variant_config
from variant_config import load_variant, save_json
importlib.reload(eng)
variant_config.EDGE_DIR = EDGE_DIR

# ---- Config ----
VARIANT = "v1_spy"
cfg        = load_variant(VARIANT)
eng.set_engine_defaults(**cfg.get("engine_defaults", {}))
RUN_DIR    = cfg["run_dir"]
TICKER     = cfg["instrument"]["ticker"]
DATA_START = cfg["instrument"]["start"]
OOS_START  = cfg["oos_start_ts"]
GAP_MIN    = cfg["params"]["gap_min_trade"]
GAP_THR    = cfg["params"]["gap_threshold"]
GAP_MAX_TF = cfg["params"]["gap_max_tf"]
ENTRY_TF   = cfg["params"]["entry_tf_min"]
MR_FILL    = cfg["params"]["mr_fill_target"]
TF_MULT    = cfg["params"]["tf_target_mult"]
MR_STOP    = cfg["params"]["mr_stop_atr_mult"]
TF_STOP    = cfg["params"]["tf_stop_atr_mult"]
TIME_EXIT  = cfg["params"].get("time_exit", "eod")
SKIP_FRI   = cfg["params"].get("skip_fridays", True)
SLIP_BPS   = cfg["params"]["slip_bps_per_side"]

In [ ]:
# ---- Data ----
intraday_raw = eng.load_data(TICKER, start=DATA_START)
daily_raw    = eng.session_daily(intraday_raw)

n_is = int((daily_raw.index < OOS_START).sum())
daily_is, daily_oos = daily_raw.iloc[:n_is].copy(), daily_raw.iloc[n_is:].copy()
oos_start = daily_oos.index[0]

intraday_dates = intraday_raw.index.normalize().tz_localize(None) if intraday_raw.index.tz is not None \
                 else intraday_raw.index.normalize()
oos_start_dt = pd.Timestamp(OOS_START)
if intraday_raw.index.tz is not None:
    oos_start_dt = oos_start_dt.tz_localize(intraday_raw.index.tz)
intraday_is  = intraday_raw[intraday_raw.index <  oos_start_dt]
intraday_oos = intraday_raw[intraday_raw.index >= oos_start_dt]

In [ ]:
# ---- Helpers ----
def _pf(t):
    if len(t) == 0: return 0.0
    p = t["pnl"].values; g = p[p > 0].sum(); l = -p[p < 0].sum()
    return float(g / l) if l > 0 else np.inf

def _sharpe(r):
    if len(r) < 2 or r.std(ddof=0) == 0: return 0.0
    return float(np.sqrt(252) * r.mean() / r.std(ddof=0))

def _mdd(eq):
    pk = np.maximum.accumulate(eq); return float(((eq - pk) / pk).min())

# ---- Baseline / IS / OOS ----
def _run(intra, daily, slip=SLIP_BPS):
    return eng.run_gap_backtest(intra, daily,
        gap_min_trade=GAP_MIN, gap_threshold=GAP_THR, gap_max_tf=GAP_MAX_TF,
        entry_tf_min=ENTRY_TF, mr_fill_target=MR_FILL, tf_target_mult=TF_MULT,
        mr_stop_atr_mult=MR_STOP, tf_stop_atr_mult=TF_STOP, time_exit=TIME_EXIT,
        skip_fridays=SKIP_FRI, slip_bps_per_side=slip)

trades_all = _run(intraday_raw, daily_raw)
trades_is  = _run(intraday_is,  daily_is)
trades_oos = _run(intraday_oos, daily_oos)
trades_atr     = trades_all.assign(pnl=trades_all["pnl_atr"], win=trades_all["pnl_atr"] > 0)
trades_oos_atr = trades_oos.assign(pnl=trades_oos["pnl_atr"], win=trades_oos["pnl_atr"] > 0)

ret_cap_is  = eng.daily_returns(daily_is,  trades_is,  "capital")
ret_cap_oos = eng.daily_returns(daily_oos, trades_oos, "capital")
ret_atr_is  = eng.daily_returns(daily_is,  trades_is,  "atr")
ret_atr_oos = eng.daily_returns(daily_oos, trades_oos, "atr")

eq_cap_oos = (1 + ret_cap_oos).cumprod(); eq_atr_oos = (1 + ret_atr_oos).cumprod()

In [ ]:
# ---- Phase 1: OOS table ----
oos_cmp = pd.DataFrame([
    {"Metric": "Trades", "IS": len(trades_is), "OOS": len(trades_oos)},
    {"Metric": "Win Rate", "IS": f"{(trades_is['win'].mean()*100 if len(trades_is) else 0):.1f}%",
                           "OOS": f"{(trades_oos['win'].mean()*100 if len(trades_oos) else 0):.1f}%"},
    {"Metric": "Profit Factor", "IS": f"{_pf(trades_is):.3f}", "OOS": f"{_pf(trades_oos):.3f}"},
    {"Metric": "Expectancy (Capital)", "IS": f"{(trades_is['pnl'].mean() if len(trades_is) else 0):.3f}%",
                                        "OOS": f"{(trades_oos['pnl'].mean() if len(trades_oos) else 0):.3f}%"},
    {"Metric": "Sharpe (Daily MTM, Capital)", "IS": f"{_sharpe(ret_cap_is):.2f}", "OOS": f"{_sharpe(ret_cap_oos):.2f}"},
    {"Metric": "Max Drawdown (Capital)", "IS": f"{_mdd((1+ret_cap_is).cumprod())*100:.2f}%",
                                          "OOS": f"{_mdd(eq_cap_oos)*100:.2f}%"},
    {"Metric": "Sharpe (ATR)", "IS": f"{_sharpe(ret_atr_is):.2f}", "OOS": f"{_sharpe(ret_atr_oos):.2f}"},
    {"Metric": "Max Drawdown (ATR)", "IS": f"{_mdd((1+ret_atr_is).cumprod())*100:.2f}%",
                                      "OOS": f"{_mdd(eq_atr_oos)*100:.2f}%"},
])
print("=" * 80); print(f"PHASE 1 — OOS Period {oos_start.date()} → {daily_oos.index[-1].date()}"); print("=" * 80)
display(oos_cmp)

In [ ]:
# ---- Phase 2: Walk-forward ----
wf = eng.walk_forward_fixed(daily_raw, intraday_raw, rules=dict(
    gap_min_trade=GAP_MIN, gap_threshold=GAP_THR, gap_max_tf=GAP_MAX_TF,
    entry_tf_min=ENTRY_TF, mr_fill_target=MR_FILL, tf_target_mult=TF_MULT,
    mr_stop_atr_mult=MR_STOP, tf_stop_atr_mult=TF_STOP, time_exit=TIME_EXIT,
    skip_fridays=SKIP_FRI, slip_bps_per_side=SLIP_BPS
), is_years=3)

mean_is = wf["ret_is_anual"].mean()
mean_oos = wf["ret_oos"].mean()
efficiency = mean_oos / mean_is if mean_is > 0 else np.nan
pct_pos = (wf["ret_oos"] > 0).mean()
worst_mdd = wf["mdd_oos"].min()
wf_pass = (efficiency >= 0.5) and (pct_pos >= 0.60) and (worst_mdd > -0.20)

print("=" * 70); print("PHASE 2 — WALK-FORWARD (fixed rules, 3y IS / 1y OOS)"); print("=" * 70)
print(f"Efficiency          : {efficiency:.3f} (≥ 0.50)")
print(f"Positive OOS ratio  : {pct_pos*100:.1f}% (≥ 60%)")
print(f"Worst OOS DD        : {worst_mdd*100:.2f}% (> −20%)")
print(f"Verdict             : {'PASSED' if wf_pass else 'FAILED'}")
display(wf)

In [ ]:
# ---- Phase 3: Block Monte Carlo ----
def mc_block(trades, n_sim=10000, block=5, seed=42):
    pnls = trades["pnl"].values / 100.0
    n = len(pnls)
    if n == 0: return {"ret_p5": 0, "ret_p50": 0, "mdd_p5": 0, "mdd_p50": 0, "mdd_worst": 0, "prob_positive": 0, "prob_ruin": 0}
    nb = int(np.ceil(n / block)); rng = np.random.default_rng(seed)
    rets, mdds = [], []
    for _ in range(n_sim):
        idx = []
        for _ in range(nb):
            s = rng.integers(0, n)
            idx.extend([(s + k) % n for k in range(block)])
        idx = idx[:n]
        sampled = pnls[idx]
        eq = np.cumprod(1 + sampled)
        rets.append(eq[-1] - 1)
        pk = np.maximum.accumulate(eq)
        mdds.append((eq / pk - 1).min())
    rets = np.array(rets); mdds = np.array(mdds)
    return {"ret_p5": float(np.percentile(rets, 5)), "ret_p50": float(np.percentile(rets, 50)),
            "mdd_p5": float(np.percentile(mdds, 5)), "mdd_p50": float(np.percentile(mdds, 50)),
            "mdd_worst": float(mdds.min()), "prob_positive": float((rets > 0).mean()),
            "prob_ruin": float((mdds <= -0.50).mean())}

mc_cap = mc_block(trades_all,          10_000, 5, 42)
mc_atr = mc_block(trades_all.assign(pnl=trades_atr["pnl_atr"]), 10_000, 5, 42)
print("=" * 70); print("PHASE 3 — BLOCK MONTE CARLO (10,000 × block=5)"); print("=" * 70)
display(pd.DataFrame([
    {"Metric": "Median return (p50)", "Capital": f"{mc_cap['ret_p50']*100:.1f}%", "ATR": f"{mc_atr['ret_p50']*100:.1f}%"},
    {"Metric": "5th pct return (p5)", "Capital": f"{mc_cap['ret_p5']*100:.1f}%",  "ATR": f"{mc_atr['ret_p5']*100:.1f}%"},
    {"Metric": "Median MDD",          "Capital": f"{mc_cap['mdd_p50']*100:.2f}%", "ATR": f"{mc_atr['mdd_p50']*100:.2f}%"},
    {"Metric": "Worst 5% MDD",        "Capital": f"{mc_cap['mdd_p5']*100:.2f}%",  "ATR": f"{mc_atr['mdd_p5']*100:.2f}%"},
    {"Metric": "Worst simulation MDD","Capital": f"{mc_cap['mdd_worst']*100:.2f}%","ATR": f"{mc_atr['mdd_worst']*100:.2f}%"},
    {"Metric": "Prob. of ruin (≤ −50%)","Capital": f"{mc_cap['prob_ruin']*100:.2f}%","ATR": f"{mc_atr['prob_ruin']*100:.2f}%"},
    {"Metric": "Prob. of positive return","Capital": f"{mc_cap['prob_positive']*100:.2f}%","ATR": f"{mc_atr['prob_positive']*100:.2f}%"},
]))

In [ ]:
# ---- Phase 4: Monkey test ----
def monkey_test(daily_df, intraday_df, trades_df, n_sim=2000, seed=42, slip_bps=SLIP_BPS):
    eligible = daily_df[(daily_df["gap_pct"].abs() >= GAP_MIN) & (daily_df["gap_pct"].abs() <= GAP_MAX_TF)].copy()
    if SKIP_FRI:
        eligible = eligible[eligible.index.dayofweek != 4]
    if len(eligible) == 0 or len(trades_df) == 0:
        return {"pf_real": 0.0, "pfs": np.array([]), "pct_pf": 0.0}